# Dog Tracking + Restricted-Area Demo

This notebook demonstrates:

**Video → YOLO dog detection → ByteTrack tracking → restricted area → dwell-time alert**

### Before running
1. Put this notebook and your dog video in the same folder.
2. Rename the video to **`dog.mp4`**.
3. Run the cells from top to bottom.
4. In the restricted-area cell, adjust the four percentage values until the red box covers the area you want to restrict.
5. The notebook will create **`output_restricted_area.mp4`**.

This demo uses a pretrained model; no model training is required.


In [ ]:
# Install required packages.
# Run this cell once. If the packages are already installed, it will finish quickly.

%pip install -q ultralytics opencv-python matplotlib imageio-ffmpeg


In [ ]:
from pathlib import Path
from collections import defaultdict
import subprocess

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display
from ultralytics import YOLO

VIDEO_PATH = "dog.mp4"
RAW_OUTPUT_PATH = "output_restricted_area_raw.mp4"
OUTPUT_PATH = "output_restricted_area.mp4"

MODEL_NAME = "yolo11n.pt"
TRACKER = "bytetrack.yaml"
CONFIDENCE = 0.25
ALERT_AFTER_SECONDS = 2.0

if not Path(VIDEO_PATH).exists():
    raise FileNotFoundError(
        f"{VIDEO_PATH} was not found. Put your dog video in the same folder as this notebook "
        "and rename it to dog.mp4."
    )

print("Video found:", VIDEO_PATH)


## 1. Load the model and preview the first video frame

The pretrained YOLO model already knows the COCO class **dog**, so this demo does not require training.


In [ ]:
model = YOLO(MODEL_NAME)

# Find the dog class ID automatically instead of hard-coding it.
DOG_CLASS_ID = None
for class_id, class_name in model.names.items():
    if class_name == "dog":
        DOG_CLASS_ID = int(class_id)
        break

if DOG_CLASS_ID is None:
    raise RuntimeError("The loaded model does not contain a 'dog' class.")

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

ok, first_frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError("Could not read the first frame.")

first_rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)

print(f"Video size: {width} x {height}")
print(f"FPS: {fps:.2f}")
print(f"Frames: {frame_count}")
print(f"Dog class ID: {DOG_CLASS_ID}")

plt.figure(figsize=(12, 7))
plt.imshow(first_rgb)
plt.title("First frame of dog.mp4")
plt.axis("off")
plt.show()


## 2. Define the restricted area

To make this easy in Jupyter, the restricted area is a **rectangle defined as a percentage of the video width and height**.

For example:

- `ZONE_LEFT = 0.55` means the left edge begins at 55% of the frame width.
- `ZONE_TOP = 0.20` means the top edge begins at 20% of the frame height.
- `ZONE_RIGHT = 0.95` means the right edge ends at 95% of the frame width.
- `ZONE_BOTTOM = 0.90` means the bottom edge ends at 90% of the frame height.

Change these four values and rerun the cell until the red rectangle covers the area you want.


In [ ]:
# EDIT THESE FOUR VALUES TO MOVE/RESIZE THE RESTRICTED AREA.
ZONE_LEFT = 0.55
ZONE_TOP = 0.20
ZONE_RIGHT = 0.95
ZONE_BOTTOM = 0.90

# Convert percentages to pixel coordinates.
x_left = int(width * ZONE_LEFT)
y_top = int(height * ZONE_TOP)
x_right = int(width * ZONE_RIGHT)
y_bottom = int(height * ZONE_BOTTOM)

restricted_polygon = np.array([
    [x_left, y_top],
    [x_right, y_top],
    [x_right, y_bottom],
    [x_left, y_bottom]
], dtype=np.int32)

# Preview the restricted area.
preview = first_frame.copy()

overlay = preview.copy()
cv2.fillPoly(overlay, [restricted_polygon], (0, 0, 255))
preview = cv2.addWeighted(overlay, 0.18, preview, 0.82, 0)
cv2.polylines(preview, [restricted_polygon], True, (0, 0, 255), 4)

preview_rgb = cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 7))
plt.imshow(preview_rgb)
plt.title("Restricted Area Preview")
plt.axis("off")
plt.show()

print("Restricted polygon coordinates:")
print(restricted_polygon.tolist())


## 3. Run dog detection, tracking, and restricted-area checking

For every video frame:

1. YOLO detects the dog.
2. ByteTrack keeps a consistent tracking ID across frames.
3. The dog's **bottom-center point** is used as its approximate floor position.
4. `cv2.pointPolygonTest()` checks whether that point is inside the restricted area.
5. If the same tracked dog stays inside for at least **2 seconds**, the video displays an alert.


In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    RAW_OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create the output video.")

# track_id -> time when the dog most recently entered the zone
entered_at = {}

# Keep recent dog positions so we can draw a trajectory.
track_history = defaultdict(list)

frame_number = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame_number += 1
    current_video_time = frame_number / fps

    # Detection + ByteTrack.
    result = model.track(
        frame,
        persist=True,
        tracker=TRACKER,
        classes=[DOG_CLASS_ID],
        conf=CONFIDENCE,
        verbose=False
    )[0]

    annotated = frame.copy()

    # Draw the restricted area.
    overlay = annotated.copy()
    cv2.fillPoly(overlay, [restricted_polygon], (0, 0, 255))
    cv2.addWeighted(overlay, 0.18, annotated, 0.82, 0, annotated)
    cv2.polylines(annotated, [restricted_polygon], True, (0, 0, 255), 3)

    cv2.putText(
        annotated,
        "RESTRICTED AREA",
        (restricted_polygon[0][0], max(30, restricted_polygon[0][1] - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        (0, 0, 255),
        2,
        cv2.LINE_AA
    )

    visible_track_ids = set()

    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        track_ids = result.boxes.id.int().cpu().tolist()
        confidences = result.boxes.conf.cpu().numpy()

        for box, track_id, confidence in zip(boxes, track_ids, confidences):
            visible_track_ids.add(track_id)

            x1, y1, x2, y2 = map(int, box)

            # Bottom-center of the dog bounding box.
            pet_point = (int((x1 + x2) / 2), int(y2))

            # Positive = inside, zero = boundary, negative = outside.
            zone_test = cv2.pointPolygonTest(
                restricted_polygon,
                pet_point,
                False
            )
            inside = zone_test >= 0

            if inside:
                if track_id not in entered_at:
                    entered_at[track_id] = current_video_time
                dwell_time = current_video_time - entered_at[track_id]
            else:
                entered_at.pop(track_id, None)
                dwell_time = 0.0

            # Save and draw trajectory.
            track_history[track_id].append(pet_point)
            if len(track_history[track_id]) > 90:
                track_history[track_id].pop(0)

            if len(track_history[track_id]) >= 2:
                trajectory = np.array(track_history[track_id], dtype=np.int32)
                cv2.polylines(
                    annotated,
                    [trajectory],
                    False,
                    (255, 255, 0),
                    2
                )

            if inside:
                box_color = (0, 0, 255)
                status = f"INSIDE | {dwell_time:.1f}s"
            else:
                box_color = (0, 255, 0)
                status = "OUTSIDE"

            # Draw dog bounding box.
            cv2.rectangle(
                annotated,
                (x1, y1),
                (x2, y2),
                box_color,
                3
            )

            # Draw the bottom-center anchor point.
            cv2.circle(
                annotated,
                pet_point,
                7,
                box_color,
                -1
            )

            label = f"Dog ID {track_id} | {confidence:.2f} | {status}"

            cv2.putText(
                annotated,
                label,
                (x1, max(30, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                box_color,
                2,
                cv2.LINE_AA
            )

            # Dwell-time alert.
            if inside and dwell_time >= ALERT_AFTER_SECONDS:
                alert_text = f"ALERT: Dog ID {track_id} in restricted area!"

                cv2.rectangle(
                    annotated,
                    (20, 55),
                    (min(width - 20, 720), 115),
                    (0, 0, 255),
                    -1
                )

                cv2.putText(
                    annotated,
                    alert_text,
                    (35, 95),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA
                )

    # If the tracker loses sight of the dog, clear its active zone timer.
    for track_id in list(entered_at.keys()):
        if track_id not in visible_track_ids:
            entered_at.pop(track_id, None)

    writer.write(annotated)

cap.release()
writer.release()

print("Tracking finished.")
print("Raw output:", RAW_OUTPUT_PATH)


## 4. Convert the result to a browser-friendly MP4

OpenCV often writes an MP4 codec that does not play reliably inside browsers.  
This cell uses FFmpeg to convert the result to H.264 when possible.


In [ ]:
import imageio_ffmpeg

ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

command = [
    ffmpeg_exe,
    "-y",
    "-i", RAW_OUTPUT_PATH,
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    OUTPUT_PATH
]

try:
    subprocess.run(
        command,
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    print("Created:", OUTPUT_PATH)
except Exception as e:
    print("H.264 conversion failed. You can still use the raw output.")
    print("Reason:", e)
    OUTPUT_PATH = RAW_OUTPUT_PATH


## 5. View the finished demo

The final video should show:

- A bounding box around the dog
- A persistent `Dog ID`
- The dog's movement path
- A red restricted area
- `INSIDE` / `OUTSIDE` status
- Time spent inside the zone
- An alert after the dwell-time threshold


In [ ]:
display(Video(OUTPUT_PATH, embed=True, width=900))


## Short explanation for your Project 1 presentation

You can say:

> This demonstration uses a pretrained YOLO model to detect the dog and ByteTrack to maintain the dog's identity across video frames. I define a restricted region as a polygon. For each tracked dog, the system uses the bottom-center of its bounding box as an estimate of its position on the floor. A point-in-polygon test determines whether the dog is inside the restricted region. If the dog remains inside for more than two seconds, the system displays an alert.

### Important limitation
This is a demonstration rather than a complete commercial pet-monitoring system. Performance can decrease with poor lighting, occlusion, very small dogs in the frame, fast movement, or unusual camera angles.
